# 01 — REDCap Data Exploration
**NANO Study | ESD Lab, University of South Carolina**

> ⚠️ **HIPAA Notice**: This notebook uses **synthetic data only**. Never load real participant data inside this notebook. All production data access goes through `src/data_ingestion/redcap_merge.py` with config-defined paths.

## Goals
1. Explore REDCap export structure and event metadata
2. Assess completeness by participant, group, and event
3. Visualize missingness patterns with naniar-style diagnostics
4. Identify cross-field inconsistencies in synthetic data


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Add repo root to path
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repo root: {repo_root}')

## 1. Generate Synthetic REDCap-Like Dataset
We generate a realistic synthetic dataset that mirrors the NANO study REDCap longitudinal structure.

In [ ]:
rng = np.random.default_rng(seed=42)
N_PARTICIPANTS = 60  # Synthetic subset
EVENTS = ['nicu_admission', 'month_1', 'month_2', 'month_3',
          'month_6', 'month_9', 'month_12', 'month_36']
GROUPS = ['ASIB', 'PT', 'TD']
GA_BINS = ['24-26w', '27-29w', '30-32w']

rows = []
for pid in range(1, N_PARTICIPANTS + 1):
    group = rng.choice(GROUPS, p=[0.20, 0.50, 0.30])
    ga_bin = rng.choice(GA_BINS, p=[0.20, 0.40, 0.40])
    ga_weeks = {'24-26w': rng.uniform(24, 27), '27-29w': rng.uniform(27, 30),
                '30-32w': rng.uniform(30, 33)}[ga_bin]
    sex = rng.choice(['M', 'F'])
    for event in EVENTS:
        # Introduce realistic missingness (20% overall, higher at later timepoints)
        event_idx = EVENTS.index(event)
        miss_prob = 0.05 + 0.04 * event_idx
        if rng.random() < miss_prob:
            continue
        rows.append({
            'nano_id': f'NANO-{pid:04d}',
            'redcap_event_name': event,
            'group': group,
            'ga_bin': ga_bin,
            'ga_weeks': round(ga_weeks, 1),
            'sex': sex,
            'birth_weight_g': int(rng.normal(1200 - 60 * event_idx, 300)),
            'nnns_attention': rng.normal(3.0, 0.8) if event in ['month_1', 'month_2', 'month_3'] else np.nan,
            'rsa': rng.normal(4.2, 0.9),
            'rmssd_ms': rng.normal(35, 12),
            'ados_css': rng.integers(1, 20) if event == 'month_36' and group == 'ASIB' else np.nan,
            'bayley_cognitive': rng.normal(90, 15) if event in ['month_12', 'month_36'] else np.nan,
            'ecg_recording_complete': rng.choice([0, 1, 2], p=[0.05, 0.15, 0.80]),
            'temp_recording_complete': rng.choice([0, 1, 2], p=[0.10, 0.20, 0.70]),
        })

df_redcap = pd.DataFrame(rows)
print(f'Synthetic dataset: {df_redcap.shape[0]} rows, {df_redcap.shape[1]} columns')
df_redcap.head()

## 2. Event Completeness by Participant Group

In [ ]:
completeness = (
    df_redcap.groupby(['group', 'redcap_event_name'])
    .size()
    .reset_index(name='n_records')
    .pivot(index='redcap_event_name', columns='group', values='n_records')
    .reindex(EVENTS)
)
print('Records per event per group:')
display(completeness)

fig, ax = plt.subplots(figsize=(10, 4))
completeness.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Visit Event')
ax.set_ylabel('N Records')
ax.set_title('NANO Study — Event Completeness by Group (Synthetic Data)')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Group')
plt.tight_layout()
plt.show()

## 3. Missingness Heatmap (naniar-style)

In [ ]:
numeric_cols = ['nnns_attention', 'rsa', 'rmssd_ms', 'ados_css', 'bayley_cognitive']
miss_matrix = df_redcap[numeric_cols].isnull().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap
sns.heatmap(miss_matrix.sample(min(200, len(miss_matrix)), random_state=0).T,
            ax=axes[0], cmap='binary', cbar=False, yticklabels=True)
axes[0].set_title('Missingness Pattern (sample of 200 rows)')
axes[0].set_xlabel('Record index')

# Percent missing bar chart
miss_pct = miss_matrix.mean() * 100
miss_pct.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_xlabel('% Missing')
axes[1].set_title('Percent Missing per Variable')
axes[1].axvline(20, color='red', linestyle='--', label='20% threshold')
axes[1].legend()

plt.tight_layout()
plt.show()
print('Missing %:')
print(miss_pct.round(1).to_string())

## 4. Cross-Field Logic Checks

In [ ]:
# Flag: TD group should not have GA < 37 weeks (all are synthetic VPT here, so this is illustrative)
# In real data, TD participants have GA >= 37 weeks
ga_flag = df_redcap[(df_redcap['group'] == 'TD') & (df_redcap['ga_weeks'] < 37)]
print(f'TD participants with GA < 37w (logic flag): {ga_flag["nano_id"].nunique()} unique IDs')

# Flag: ADOS scores should only appear at month_36 and only for ASIB group
ados_flag = df_redcap[
    df_redcap['ados_css'].notna() &
    ((df_redcap['redcap_event_name'] != 'month_36') | (df_redcap['group'] != 'ASIB'))
]
print(f'ADOS CSS scores at unexpected event/group: {len(ados_flag)} records')

# Flag: Extreme RSA values
rsa_flag = df_redcap[df_redcap['rsa'] < 0]
print(f'Negative RSA values (impossible): {len(rsa_flag)} records')

## 5. Summary
- Synthetic dataset demonstrates expected REDCap longitudinal structure
- Event attrition increases at later timepoints (expected)
- ADOS/Bayley variables have high missingness at early timepoints by design
- Cross-field logic checks identify structural data quality issues before analysis

**Next steps**: See `02_ecg_preprocessing_walkthrough.ipynb` for physiological signal processing.